**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Intro to MATLAB

> ⚠️ **Draft — code not machine-verified.** MATLAB code below is shown in fenced blocks and must be run in MATLAB/Octave; it has not been executed by an automated check. An instructor should run each block once before teaching. Remove this banner after that pass.

MATLAB is the language much of signal processing *thinks* in: matrices are the native type, the toolboxes mirror the textbook, and UF provides licenses to students. This workshop parallels [Intro to Python](../Intro_Python/Intro_Python.ipynb) — same concepts, MATLAB idiom — then reproduces core results from [Foundations of Signal Processing](../../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb) and [Filter Design](../../Intro_DSP/Filter_Design.ipynb).

**Setup:** UF students — MATLAB via [UF Apps](https://info.apps.ufl.edu/) or a student license; or use [GNU Octave](https://octave.org/) (free, runs nearly everything here).

---
### 🕐 Session 1 of 2 — *MATLAB Fundamentals* (~35 min)
**Goal:** matrices as the native type; indexing, scripts vs functions, plotting.
**Builds on:** [Intro to Python](../Intro_Python/Intro_Python.ipynb) (concept-for-concept parallel). &nbsp; **Feeds into:** Session 2 (signal processing).

---

## 1. Everything Is a Matrix

💡 **Intuition.** In Python, NumPy arrays are a library; in MATLAB they're the *language*. `5` is a 1×1 matrix. This inversion explains all the syntax: `*` is matrix multiply and you ask for elementwise with `.*` — exactly opposite to NumPy's convention. Most porting bugs between the two languages are this one convention flip.

```matlab
A = [1 2 3; 4 5 6];      % 2x3 matrix — semicolon = new row
x = 0:0.1:1;             % range with step (inclusive!)
t = linspace(0, 1, 500); % 500 points, like np.linspace

size(A)                  % 2 3
A'                       % transpose
A(1, :)                  % first ROW  — indexing is 1-based, round brackets
A(:, 2)                  % second column
A(end, end)              % last element: 'end' is a keyword

B = A .* A;              % elementwise  (.* ./ .^)
C = A * A';              % MATRIX multiply (2x3 * 3x2 = 2x2)
```

The three habits that prevent 90% of Python↔MATLAB confusion:

| | MATLAB | NumPy |
|---|---|---|
| Indexing | `A(1,1)`, 1-based | `A[0,0]`, 0-based |
| Elementwise / matrix `*` | `.*` / `*` | `*` / `@` |
| Range end | `0:5` **includes** 5 | `range(0,5)` excludes 5 |

## 2. Scripts, Functions, Plotting

```matlab
% functions can live at the end of a script file (R2016b+) or in their own file
function y = db(power, ref)
    if nargin < 2, ref = 1.0; end        % default argument, MATLAB-style
    y = 10 * log10(power / ref);
end
```

```matlab
fs = 500;  t = 0:1/fs:1-1/fs;
clean = sin(2*pi*5*t);
noisy = clean + 0.4*randn(size(t));

figure;
plot(t, noisy, 'DisplayName', 'noisy'); hold on;
plot(t, clean, 'LineWidth', 2, 'DisplayName', 'clean');
xlabel('time [s]'); ylabel('amplitude');
title('A 5 Hz sine in noise'); legend; grid on;
```

Same figure as [Intro to Python §5](../Intro_Python/Intro_Python.ipynb) — `hold on` is MATLAB's
"keep drawing on this axes."


---
### 🕐 Session 2 of 2 — *Signal Processing in MATLAB* (~40 min)
**Goal:** `fft`, `filter`, `freqz` — reproduce the Python DSP results in MATLAB idiom.
**Builds on:** Session 1; [DSP Workshop 1](../../Intro_DSP/README.md).

---

## 3. The Spectrum

💡 **Intuition.** `fft` is the same radix-2 machine derived in [Foundations Session 5](../../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb) — MATLAB just returns the *two-sided* spectrum, so the standard recipe keeps the first half and doubles it (except DC and Nyquist).

```matlab
X = fft(noisy);
N = numel(noisy);
f = (0:N/2) * fs / N;                    % one-sided frequency axis
P = abs(X(1:N/2+1)) / N;
P(2:end-1) = 2 * P(2:end-1);             % fold negative frequencies in

figure; plot(f, P); xlim([0 50]);
xlabel('frequency [Hz]'); ylabel('|X(f)|');
title('The 5 Hz peak pops right out of the noise');
```

## 4. Filters (Signal Processing Toolbox)

The [Filter Design](../../Intro_DSP/Filter_Design.ipynb) workflow, line for line:

```matlab
% FIR low-pass: 101 taps, 100 Hz cutoff at fs = 1000  (≈ scipy firwin)
fs = 1000;
taps = fir1(100, 100/(fs/2));            % order 100 → 101 taps; freqs normalized to Nyquist!
freqz(taps, 1, 2048, fs);                % magnitude AND phase — read both

% IIR Butterworth, order 4  (≈ scipy butter)
[b, a] = butter(4, 100/(fs/2));
fvtool(b, a);                            % interactive response explorer
zplane(b, a);                            % poles inside the unit circle ⇒ stable

% zero-phase filtering (≈ scipy filtfilt)
y = filtfilt(b, a, noisy);
```

⚠️ The classic trap: MATLAB normalizes frequencies to the **Nyquist** rate (`Wn = f/(fs/2)`),
while `scipy` (with the `fs=` argument) takes plain Hz. Wrong normalization = silently wrong
cutoff — check `freqz` *every time*.

## 5. Conclusion

You can now move between the curriculum's two dialects: NumPy for glue-and-scale, MATLAB for toolbox-and-textbook. The concepts — spectra, taps, poles — are identical; only conventions differ, and you know the three that bite.

---
## Where next

- [Filter Design](../../Intro_DSP/Filter_Design.ipynb) — the Python mirror of §4.
- [Foundations of Signal Processing](../../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb) — what `fft` actually computes.